# MS COCO Multi-Label Image Classification — Final Submission

This notebook documents our experiments for the **MS COCO multi-label image classification challenge** with 80 object categories. Each image may contain multiple categories, so the pipeline uses sigmoid probabilities and class-specific decision thresholds.

Our experiments covered architecture comparisons and optimization of ViT-Large at **384 × 384** with whole-image squash preprocessing, learning-rate tuning, layer-wise learning-rate decay, and exponential moving averages (EMA).

The best configuration in the six-trial LR × layer-decay search was **Trial 5**. The saved checkpoint metadata records LR = `1.5e-4`, layer decay = `0.75`, best validation epoch = `3`, validation half-split F1 = `0.74238`, and EMA weights. The reported hidden-test leaderboard result associated with the final submission was **F1 = 0.7723**.

### Reported final leaderboard metrics

| Metric | Score |
|---|---:|
| Accuracy | 0.8224 |
| Precision | 0.7483 |
| Recall | 0.7979 |
| **F1-score** | **0.7723** |

**Metric note:** The leaderboard score is a hidden-test result and is not directly comparable to the validation half-split F1 in the trial table.


## 1. Colab environment setup

The dataset is copied once from Google Drive to the fast local disk of the VM (`/content/data`). A `.zip` file is copied with a resumable routine (continues after a dropped connection, verifies the final size) and unzipped locally; a marker file makes re-running the cell cheap.

In [ ]:
# =============================================================================
# 1. COLAB ENVIRONMENT SETUP
# =============================================================================
!pip install -q timm

import os
import shutil
import subprocess
import time
from pathlib import Path

import torch
from google.colab import drive

# --- 1.1 GPU check -----------------------------------------------------------
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Use Runtime > Change runtime type > A100 GPU.")

# --- 1.2 Mount Google Drive ---------------------------------------------------
drive.mount('/content/drive')

# --- 1.3 Paths (edit these to match your Drive layout) ------------------------
DRIVE_ZIP  = Path('/content/drive/MyDrive/ms-coco.zip')          # option A: a single zip file (recommended)
DRIVE_DATA = Path('/content/drive/MyDrive/ms-coco')              # option B: an unzipped folder
OUTPUT_DIR = Path('/content/drive/MyDrive/vit_coco_output_v2')   # results are written below this folder (Drive)
LOCAL_ROOT = Path('/content/data')                               # fast local disk (erased on runtime reset)


def robust_copy(src, dst, chunk=64 * 1024 * 1024, retries=20):
    """Copy a large file from the (sometimes unstable) Drive mount to local disk.

    If the connection drops, the copy continues from the bytes already written, and the
    final size is verified so a truncated file is never used silently.
    """
    src, dst = str(src), str(dst)
    total = os.path.getsize(src)
    for attempt in range(retries + 1):
        done = os.path.getsize(dst) if os.path.exists(dst) else 0
        if done >= total:
            break
        try:
            with open(src, 'rb') as fi, open(dst, 'ab' if done else 'wb') as fo:
                fi.seek(done)
                while True:
                    buf = fi.read(chunk)
                    if not buf:
                        break
                    fo.write(buf)
        except OSError as e:
            done = os.path.getsize(dst) if os.path.exists(dst) else 0
            print(f"Connection dropped ({done / 1e9:.2f}/{total / 1e9:.2f} GB), retry {attempt + 1}: {e}")
            time.sleep(5)
    assert os.path.getsize(dst) == total, "Copy is incomplete - please re-run this cell."


# --- 1.4 Drive -> local disk (skipped if it was already done in this session) -
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
done_marker = LOCAL_ROOT / '.copy_done'      # only created after a fully successful copy

if done_marker.exists():
    print("Dataset already present on the local disk - skipping the copy.")
else:
    if DRIVE_ZIP.exists():
        local_zip = Path('/content/ms-coco.zip')
        print("Copying the zip file to the local disk ...")
        robust_copy(DRIVE_ZIP, local_zip)
        print("Unzipping (local) ...")
        subprocess.run(['unzip', '-q', '-o', str(local_zip), '-d', str(LOCAL_ROOT)], check=True)
        local_zip.unlink()
    elif DRIVE_DATA.exists():
        print(f"Copying the folder (this takes a while): {DRIVE_DATA}")
        shutil.copytree(DRIVE_DATA, LOCAL_ROOT / 'ms-coco', dirs_exist_ok=True)
    else:
        raise FileNotFoundError(
            f"Dataset not found on Drive.\n  zip:    {DRIVE_ZIP}\n  folder: {DRIVE_DATA}\nPlease check the paths."
        )
    done_marker.touch()
    print("Copy finished.")

# --- 1.5 Show the directory structure (depth <= 3) -----------------------------
for dirpath, dirnames, _ in os.walk(LOCAL_ROOT):
    depth = len(Path(dirpath).relative_to(LOCAL_ROOT).parts)
    if depth > 3:
        dirnames[:] = []
        continue
    print(Path(dirpath).relative_to(LOCAL_ROOT) if depth else '.')

# --- 1.6 Variables used by the rest of the notebook ----------------------------
DATA_ROOT = LOCAL_ROOT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# An A100 Colab VM usually has about 12 CPU cores. The augmentation pipeline (RandAugment on PIL images)
# is CPU-bound, so use many workers - but never more than there are cores.
NUM_WORKERS = min(12, os.cpu_count() or 2)
print("CPU cores:", os.cpu_count(), "| NUM_WORKERS =", NUM_WORKERS)


### Experimental environments

Our experiments were conducted in two different computing environments:

| Experimental branch | Environment | GPU |
|---|---|---|
| Architecture comparison, loss functions and ensembling | Local Ubuntu workstation | NVIDIA RTX 4060 Laptop GPU (8 GB) |
| ViT-Large 384×384 optimization | Google Colab | NVIDIA A100 |





## 2. Configuration and LR × layer-decay search

The search kept the 384 × 384 whole-image squash pipeline and other training settings fixed, changing only learning rate (LR) and layer-wise learning-rate decay.

| Setting | Value |
|---|---:|
| Model | ViT-Large, `vit_large_patch16_224.augreg_in21k_ft_in1k` |
| Input resolution | 384 × 384 |
| Preprocessing | Resize entire image directly to 384 × 384 (squash) |
| Validation split | 80% train / 20% validation |
| Batch size | 16 |
| Epochs per trial | 6 |
| LR candidates | `7.5e-5`, `1e-4`, `1.5e-4` |
| Layer-decay candidates | `0.75`, `0.85` |
| Weight decay | 0.05 |
| Label smoothing | 0.10 |
| Positive-weight strength / cap | 0.5 / 20.0 |
| Drop rate / drop-path rate | 0.10 / 0.20 |
| EMA decay | 0.9996 |
| Optimizer / scheduler | AdamW / OneCycleLR |
| TTA during validation and test | Horizontal-flip TTA enabled |

### Recorded results

The table reports each trial's best **validation half-split F1**, which was the criterion used to select the best trial. “Full F1” and “macro-F1” are the values printed at that same best-half-split epoch.

| Trial | LR | Layer decay | Best epoch | Half-split F1 | Full F1 at selected epoch | Macro-F1 at selected epoch |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 7.5e-5 | 0.75 | 6 | 0.73608 | 0.7766 | 0.8051 |
| 2 | 7.5e-5 | 0.85 | 4 | 0.71678 | 0.7722 | 0.8112 |
| 3 | 1.0e-4 | 0.75 | 4 | 0.73727 | 0.7687 | 0.8034 |
| 4 | 1.0e-4 | 0.85 | 3 | 0.73179 | 0.7651 | 0.8017 |
| **5** | **1.5e-4** | **0.75** | **3** | **0.74238** | **0.7690** | **0.8021** |
| 6 | 1.5e-4 | 0.85 | 5 | 0.72823 | 0.7678 | 0.8114 |

Trial 5 achieved the highest validation half-split F1 and was selected as the best trial. The previous 384 × 384 squash baseline (LR = `1e-4`, layer decay = `0.80`) had a reported hidden-test F1 of `0.7693`; this hidden-test score should not be directly compared with the validation values above.


In [ ]:
# =============================================================================
# 2. CONFIGURATION
# =============================================================================
from __future__ import annotations

import copy
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import timm
import torch
from PIL import Image
from timm.data import create_transform, resolve_data_config
from timm.optim import create_optimizer_v2
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler

NUM_CLASSES = 80

MODEL_NAME = "vit_large_patch16_224.augreg_in21k_ft_in1k"
RESOLUTION = 384

VALIDATION_FRACTION = 0.20
SEED = 42

# Fixed training settings
BATCH_SIZE = 16
EPOCHS = 6
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.10
POS_WEIGHT_STRENGTH = 0.5
POS_WEIGHT_CAP = 20.0
DROP_RATE = 0.10
DROP_PATH_RATE = 0.20
EMA_DECAY = 0.9996

# Only these two parameters are tuned.
LR_CANDIDATES = (7.5e-5, 1.0e-4, 1.5e-4)
LAYER_DECAY_CANDIDATES = (0.75, 0.85)

# Existing 384+squash baseline for reference:
BASELINE_LR = 1.0e-4
BASELINE_LAYER_DECAY = 0.80
BASELINE_TEST_F1 = 0.7693

RARE_SAMPLING_ALPHA = 0.0
USE_TTA = True
EVAL_RAW = False
MIN_EVAL_EPOCH = 2
RESUME_EVERY = 0

RUN_ROOT = Path(OUTPUT_DIR) / "vit_large_384_lr_layer_decay_a100"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = RUN_ROOT / "best_vit_large_384_lr_layer_decay_a100.pt"
RESULTS_PATH = RUN_ROOT / "lr_layer_decay_results.json"
HISTORY_PATH = RUN_ROOT / "all_trial_histories.json"

CLASS_NAMES = (
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush",
)
assert len(CLASS_NAMES) == NUM_CLASSES

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


## 3. Data: datasets and path discovery

* **Training images** come with one annotation file per image (`<image id>.cls`) containing the ids of the categories present; they become an 80-dimensional **multi-hot vector**.
* **Test images** have no labels; the dataset returns the image and its file name (without extension), the key of the submission JSON.

In [ ]:
# =============================================================================
# 3. DATASETS AND PATH DISCOVERY
# =============================================================================
class CocoTrainDataset(Dataset):
    """Labelled images. Returns (image tensor, 80-dim multi-hot target)."""

    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.transform = transform

        # One <name>.cls file per image; the sorted file stems define a fixed image order.
        self.names = sorted(p.stem for p in self.label_dir.glob("*.cls"))
        if not self.names:
            raise FileNotFoundError(f"No .cls files found in {self.label_dir}")

        # Pre-compute the multi-hot targets once: shape (num_images, 80).
        self.targets = torch.zeros((len(self.names), NUM_CLASSES), dtype=torch.float32)
        for i, name in enumerate(self.names):
            labels = [int(x) for x in (self.label_dir / f"{name}.cls").read_text().split()]
            for cls_id in labels:
                if 0 <= cls_id < NUM_CLASSES:
                    self.targets[i, cls_id] = 1.0

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = Image.open(self.image_dir / f"{name}.jpg").convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, self.targets[index]


class CocoTestDataset(Dataset):
    """Unlabelled test images. Returns (image tensor, file stem used as the submission key)."""

    def __init__(self, image_dir, transform):
        self.paths = sorted(Path(image_dir).glob("*.jpg"))
        if not self.paths:
            raise FileNotFoundError(f"No test images found in {image_dir}")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        return self.transform(Image.open(path).convert("RGB")), path.stem


def dataset_paths(data_root):
    """Return (train image dir, label dir, test image dir); several folder layouts are tried."""
    root = Path(data_root)
    candidates = []
    for dataset_root in [root, root / "ms-coco", root / "ms_coco"]:
        candidates.extend([
            (dataset_root / "images" / "train", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            (dataset_root / "images" / "train", dataset_root / "labels", dataset_root / "images" / "test"),
            (dataset_root / "images" / "train2017", dataset_root / "labels" / "train", dataset_root / "images" / "test"),
            (dataset_root / "train", dataset_root / "labels", dataset_root / "test"),
        ])
    for paths in candidates:
        if all(Path(p).is_dir() for p in paths):
            return paths
    raise FileNotFoundError(f"Could not find dataset below {root}")


### 3.1 Dataset exploration and class imbalance

Before training the models, we analyzed the dataset to understand its multi-label structure and class distribution.

The dataset contains:

| Property | Value |
|---|---:|
| Training images | 65,000 |
| Test images | 4,952 |
| Object categories | 80 |
| Average labels per training image | 2.93 |
| Minimum labels per image | 1 |
| Maximum labels per image | 18 |

Each training image is associated with an 80-dimensional multi-hot target vector, where several classes can be active simultaneously.

#### Class frequency distribution

The dataset is strongly imbalanced. Some categories are represented by thousands of images, while others appear only a few hundred times.

| Most frequent classes | Images | Rarest classes | Images |
|---|---:|---|---:|
| Person | 35,494 | Hair drier | 102 |
| Chair | 7,043 | Toaster | 117 |
| Car | 6,811 | Parking meter | 396 |
| Dining table | 6,546 | Bear | 516 |
| Cup | 5,057 | Scissors | 555 |

The most frequent category, *person*, appears approximately 348 times more often than *hair drier*, the rarest category.

#### Observations

This imbalance has important consequences for training and evaluation.

A model can achieve good overall accuracy by recognizing common classes while performing poorly on rare ones. This is especially relevant to our challenge because the official F1 metric gives greater weight to less frequent categories.

These observations motivated experiments with weighted binary cross-entropy, focal loss, asymmetric loss, class-specific threshold calibration and alternative sampling strategies.

However, the experiments also showed that increasing the importance of rare classes does not necessarily improve generalization, particularly when it produces additional false positives.


## 4. Model, metrics, prediction, EMA

### Metrics

The network outputs one probability per class; a class is predicted when its probability reaches the class threshold.

* **macro-F1** – mean of the 80 per-class F1 scores (every class counts equally).
* **leaderboard-style F1** – per-class precision and recall are averaged with weights proportional to `1 / (number of positives of the class)`, and F1 is the harmonic mean of those two weighted averages. Rare classes therefore dominate it.
* **Threshold tuning** for the leaderboard-style metric is a coordinate ascent: for each class in turn the threshold that maximises the weighted F1 is chosen while all other thresholds stay fixed.
* **Half-split estimate** (`score_probs`): thresholds tuned on the whole validation set give an *optimistic* score (they are scored on the images they were tuned on). The half-split estimate tunes the thresholds on one random half and scores them on the other half (and vice versa), which is a fairer estimate. It is used to **select the best epoch**.

### Prediction

`predict_probs` runs the model in bfloat16 autocast and can average the predictions of the image and its horizontal flip (test-time augmentation).

In [ ]:
# =============================================================================
# 4. MODEL, METRICS, PREDICTION, EMA
# =============================================================================
def make_model(model_name, device, resolution=None, drop_rate=None, drop_path_rate=None, pretrained=True):
    """Create a timm model with a new 80-way classification head.

    resolution     : input size; timm interpolates the pretrained position embeddings to this size.
    drop_path_rate : stochastic depth (only active in training mode).
    pretrained     : False when the weights will be overwritten from our own checkpoint (saves a download).
    """
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        img_size=RESOLUTION if resolution is None else resolution,
        drop_rate=DROP_RATE if drop_rate is None else drop_rate,
        drop_path_rate=DROP_PATH_RATE if drop_path_rate is None else drop_path_rate,
    )
    return model.to(device)


def make_pos_weight(targets, strength=0.5, cap=20.0):
    """Per-class positive weights for BCEWithLogitsLoss. targets: (num_train_images, 80) multi-hot matrix."""
    positives = targets.sum(0).clamp_min(1)
    ratio = ((len(targets) - positives) / positives).clamp(max=cap)   # negatives per positive, capped
    return 1 + strength * (ratio - 1)                                 # blend between 1 (no weight) and the ratio


def _class_counts(probs, targets, thresholds):
    """True positives, false positives and positives per class."""
    pred = probs >= thresholds.unsqueeze(0)
    y = targets.bool()
    tp = (pred & y).sum(0).float()
    fp = (pred & ~y).sum(0).float()
    return tp, fp, y.sum(0).float()


def macro_f1(probs, targets, thresholds):
    """Plain average of the 80 per-class F1 scores."""
    tp, fp, total = _class_counts(probs, targets, thresholds)
    denom = 2 * tp + fp + (total - tp)                   # 2TP + FP + FN
    f1 = torch.zeros_like(tp)
    valid = denom > 0
    f1[valid] = 2 * tp[valid] / denom[valid]
    return f1.mean().item()


def _class_weights(total):
    """Inverse-frequency class weights (normalised to sum to 1); classes without positives get 0."""
    w = torch.where(total > 0, 1.0 / total.clamp_min(1), torch.zeros_like(total))
    return w / w.sum()


def leaderboard_f1(probs, targets, thresholds):
    """Leaderboard-style metric. Returns (F1, weighted precision, weighted recall)."""
    tp, fp, total = _class_counts(probs, targets, thresholds)
    prec = torch.where(tp > 0, tp / (tp + fp).clamp_min(1), torch.zeros_like(tp))
    rec = torch.where(tp > 0, tp / total.clamp_min(1), torch.zeros_like(tp))
    w = _class_weights(total)
    P, R = (w * prec).sum(), (w * rec).sum()
    return (2 * P * R / (P + R).clamp_min(1e-12)).item(), P.item(), R.item()


def calibrate_thresholds_lb(probs, targets, rounds=3, init=0.35):
    """Per-class thresholds that maximise the leaderboard-style F1 (coordinate ascent).

    Only one class' precision / recall changes while its threshold is varied, so the global weighted
    precision / recall can be updated incrementally, which keeps the search fast.
    """
    grid = torch.arange(0.05, 0.86, 0.01)                # candidate thresholds
    y = targets.bool()
    total = y.sum(0).float()
    w = _class_weights(total)

    # Precision / recall of every class for every candidate threshold: shape (80, num_candidates).
    prec = torch.zeros(NUM_CLASSES, len(grid))
    rec = torch.zeros_like(prec)
    for c in range(NUM_CLASSES):
        p = probs[:, c:c + 1] >= grid.unsqueeze(0)       # (N, num_candidates)
        yc = y[:, c:c + 1]
        tp = (p & yc).sum(0).float()
        fp = (p & ~yc).sum(0).float()
        prec[c] = torch.where(tp > 0, tp / (tp + fp).clamp_min(1), torch.zeros_like(tp))
        rec[c] = torch.where(tp > 0, tp / total[c].clamp_min(1), torch.zeros_like(tp))

    ar = torch.arange(NUM_CLASSES)
    idx = torch.full((NUM_CLASSES,), int((grid - init).abs().argmin()))   # start every class near `init`
    for _ in range(rounds):
        for c in range(NUM_CLASSES):
            P = (w * prec[ar, idx]).sum()
            R = (w * rec[ar, idx]).sum()
            Pc = P - w[c] * prec[c, idx[c]] + w[c] * prec[c]    # replace class c's contribution
            Rc = R - w[c] * rec[c, idx[c]] + w[c] * rec[c]
            idx[c] = (2 * Pc * Rc / (Pc + Rc).clamp_min(1e-12)).argmax()
    thresholds = grid[idx]
    return thresholds, leaderboard_f1(probs, targets, thresholds)[0]


def score_probs(probs, targets):
    """Returns (thresholds tuned on all images, optimistic F1, half-split F1).

    The half-split F1 tunes the thresholds on a random half and scores them on the other half (both
    directions, averaged). It is the less optimistic number and is used to pick the best epoch.
    """
    thresholds, f1_full = calibrate_thresholds_lb(probs, targets)
    n = len(targets)
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(0))   # fixed split => comparable across epochs
    half_a, half_b = perm[: n // 2], perm[n // 2:]
    honest = []
    for fit, test in ((half_a, half_b), (half_b, half_a)):
        th, _ = calibrate_thresholds_lb(probs[fit], targets[fit])
        honest.append(leaderboard_f1(probs[test], targets[test], th)[0])
    return thresholds, f1_full, sum(honest) / 2


def per_class_report(probs, targets, thresholds, top=12):
    """Print the classes that cost the most leaderboard points (class weight x (1 - F1))."""
    tp, fp, total = _class_counts(probs, targets, thresholds)
    prec = torch.where(tp > 0, tp / (tp + fp).clamp_min(1), torch.zeros_like(tp))
    rec = torch.where(tp > 0, tp / total.clamp_min(1), torch.zeros_like(tp))
    f1 = 2 * prec * rec / (prec + rec).clamp_min(1e-12)
    w = _class_weights(total)
    loss = w * (1 - f1)
    share = loss / loss.sum()
    print(f"{'class':<15}{'F1':>7}{'prec':>7}{'rec':>7}{'val pos':>9}{'weight':>9}{'loss share':>12}")
    for c in loss.argsort(descending=True)[:top].tolist():
        print(f"{CLASS_NAMES[c]:<15}{f1[c]:7.3f}{prec[c]:7.3f}{rec[c]:7.3f}{int(total[c]):9d}"
              f"{100 * w[c]:8.1f}%{100 * share[c]:11.1f}%")


@torch.inference_mode()
def predict_probs(model, loader, device, amp=True, tta=False):
    """Run the model over a labelled loader. Returns (probabilities, targets) on the CPU, shape (N, 80).

    amp : bfloat16 autocast (fast on the A100).
    tta : average the sigmoid outputs of the image and its horizontal flip.
    """
    model.eval()
    all_probs, all_targets = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp and device.type == "cuda"):
            probs = model(images).float().sigmoid()
            if tta:
                probs = 0.5 * (probs + model(images.flip(-1)).float().sigmoid())   # flip along the width axis
        all_probs.append(probs.cpu())
        all_targets.append(labels.cpu())
    return torch.cat(all_probs), torch.cat(all_targets)


class EMA:
    """Exponential moving average of the model weights: ema = d * ema + (1 - d) * weights after every step."""

    def __init__(self, model, decay=0.9996):
        self.module = copy.deepcopy(model).eval()        # separate copy that is never trained directly
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.steps = 0
        state = self.module.state_dict()
        self.float_keys = [k for k, v in state.items() if v.dtype.is_floating_point]
        self.other_keys = [k for k, v in state.items() if not v.dtype.is_floating_point]

    @torch.no_grad()
    def update(self, model):
        self.steps += 1
        # Warm-up: at the beginning use a smaller decay so the average follows the model quickly.
        d = min(self.decay, (1 + self.steps) / (10 + self.steps))
        model_state, ema_state = model.state_dict(), self.module.state_dict()
        ema_tensors = [ema_state[k] for k in self.float_keys]
        model_tensors = [model_state[k].detach() for k in self.float_keys]
        torch._foreach_mul_(ema_tensors, d)                       # fused update: one call for all tensors
        torch._foreach_add_(ema_tensors, model_tensors, alpha=1 - d)
        for k in self.other_keys:                                 # integer buffers are copied
            ema_state[k].copy_(model_state[k])


def atomic_save(obj, path):
    """torch.save through a temporary file, so an interrupted save never corrupts `path`."""
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


### 4.1 Evaluation strategy and lessons from threshold calibration

During the architecture comparison experiments, We investigated how decision thresholds affect multi-label classification performance.

Unlike standard macro-F1, which averages the F1 scores of individual classes, the official challenge metric first computes inverse-frequency-weighted precision and recall, then combines them using their harmonic mean.


#### Threshold calibration experiments

We compared several prediction strategies:

- A fixed threshold of 0.5 for all 80 classes.
- A single global threshold optimized on validation data.
- Individual thresholds calibrated separately for each class.
- Threshold optimization targeting the leaderboard-style metric.

Class-specific calibration generally improved validation performance. However, directly optimizing all 80 thresholds for the leaderboard-style F1 sometimes produced large validation gains that did not transfer to the hidden test set.



## 5. Data pipeline and diagnosis of the previous checkpoint

### 5.1 Data pipeline

The labelled images are split once into training and validation parts (same seed and fraction as before, so the **validation images are the same as in the previous run**). The preprocessing configuration is read from the model architecture and then forced to 384 x 384. Training images use timm's augmentation (random-resized crop, horizontal flip, RandAugment, random erasing); validation images use whole-image resize (squash) to 384 x 384; no center crop is applied.

In [ ]:
# =============================================================================
# 5.1 DATA PIPELINE
# =============================================================================
seed_everything(SEED)
import torchvision.transforms as T
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_dir, label_dir, test_dir = dataset_paths(DATA_ROOT)

# ---- Preprocessing configuration ----------------------------------------------
# The architecture is built WITHOUT pretrained weights (no download); only its default preprocessing
# configuration (normalisation mean/std, interpolation, crop fraction, ...) is needed here.
probe_model = make_model(MODEL_NAME, torch.device("cpu"), pretrained=False)
data_config = resolve_data_config(probe_model.pretrained_cfg, model=probe_model)
del probe_model
data_config["input_size"] = (3, RESOLUTION, RESOLUTION)     # force the fine-tuning resolution

# ---- Train / validation split (fixed by the seed) ------------------------------
base_dataset = CocoTrainDataset(image_dir, label_dir)        # used for the targets only (no transform)
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(base_dataset), generator=generator).tolist()
split = int(len(base_dataset) * (1 - VALIDATION_FRACTION))
train_indices, val_indices = indices[:split], indices[split:]
print(f"Total: {len(base_dataset)} | train: {len(train_indices)} | validation: {len(val_indices)}")

# ---- Transforms ------------------------------------------------------------------
train_transform = create_transform(
    **data_config, is_training=True,
    auto_augment="rand-m9-mstd0.5-inc1",    # RandAugment: magnitude 9, magnitude noise 0.5
    re_prob=0.25, re_mode="pixel",          # random erasing with probability 0.25
)

# IMPORTANT: keep training augmentation unchanged.
# For validation/inference, use the whole image resized directly to 384 x 384.
# This removes center-cropping, which can discard small objects near image borders.
squash_size = data_config["input_size"][-1]
squash_mean = data_config["mean"]
squash_std = data_config["std"]

val_transform = T.Compose([
    T.Resize(
        (squash_size, squash_size),
        interpolation=T.InterpolationMode.BICUBIC,
    ),
    T.ToTensor(),
    T.Normalize(mean=squash_mean, std=squash_std),
])

train_dataset = Subset(CocoTrainDataset(image_dir, label_dir, train_transform), train_indices)
val_dataset = Subset(CocoTrainDataset(image_dir, label_dir, val_transform), val_indices)

# ---- Data loaders ------------------------------------------------------------------
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                     persistent_workers=(NUM_WORKERS > 0))
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = 4     # batches prepared in advance per worker

if RARE_SAMPLING_ALPHA > 0:
    # Rare-class oversampling: images that contain rare classes are drawn more often.
    train_targets = base_dataset.targets[train_indices]
    class_weight = train_targets.sum(0).clamp_min(1).pow(-RARE_SAMPLING_ALPHA)    # rarer class -> larger weight
    image_weight = (train_targets * class_weight).max(1).values                    # strongest rare class of the image
    image_weight[image_weight == 0] = class_weight.min()                          # images without any object
    sampler = WeightedRandomSampler(image_weight, num_samples=len(train_indices), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **loader_kwargs)
    print(f"Rare-class oversampling ON (alpha={RARE_SAMPLING_ALPHA})")
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kwargs)

val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, **loader_kwargs)   # larger batch: no gradients
print(f"Steps per epoch: {len(train_loader)}")


### 5.1.1 Comparison of image preprocessing strategies

During our experiments, we investigated different image preprocessing strategies, particularly because multi-label classification requires detecting all relevant objects, including those near image boundaries.

We experimented with **letterbox resizing**, which preserves the full image and its original aspect ratio by adding padding.

The ViT-Large experiments followed a different approach: **whole-image squash resizing** to 384 × 384. This preserves the entire image without padding, although it can change the original aspect ratio.

| Strategy | Full image preserved | Aspect ratio preserved | Used in |
|---|---|---|---|
| Center crop | No | Yes, within the retained crop | Initial ConvNeXt and ViT-Base experiments |
| Letterbox | Yes | Yes | Multi-view ensemble and ML-Decoder experiments |
| Squash 384 × 384 | Yes | No | ViT-Large validation and test inference |
.

These experiments illustrate an important trade-off: preserving the complete image can help retain small or peripheral objects, but the choice between padding and geometric distortion may affect how well the pretrained model recognizes them.


### 5.2 Diagnose the previous checkpoint (run this before training)

The previous run reported a validation macro-F1 of 0.80 but scored ~0.72 on the server. This cell scores the **same checkpoint on the same validation images with the leaderboard-style metric**:

* If the validation leaderboard-F1 is close to the server score, the gap came from the *metric*, not from over-fitting; the remaining headroom is in the rare classes listed in the table.
* If the validation leaderboard-F1 is much higher than the server score, the thresholds / selection over-fit the validation set.

The table lists the classes that cost the most points (class weight x (1 - F1)).

In [ ]:
# =============================================================================
# 5.2 DIAGNOSE THE PREVIOUS CHECKPOINT WITH THE LEADERBOARD-STYLE METRIC
# =============================================================================
def evaluate_checkpoint(path, amp=False, tta=False):
    """Score a saved checkpoint on the validation images with macro-F1 and the leaderboard-style F1."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model = make_model(
        checkpoint["model_name"], device,
        resolution=checkpoint["resolution"],
        drop_rate=checkpoint.get("hyperparameters", {}).get("drop_rate", 0.0),
        drop_path_rate=0.0,
        pretrained=False,                               # weights come from the checkpoint
    )
    model.load_state_dict(checkpoint["state_dict"])
    # fp32 evaluation by default = the precision in which the checkpoint's thresholds were tuned.
    probs, targets = predict_probs(model, val_loader, device, amp=amp, tta=tta)
    del model
    torch.cuda.empty_cache()

    # The old checkpoint thresholds may have been tuned with center-crop.
    # Therefore, the saved-threshold score is diagnostic only; retuned thresholds
    # below are the fair comparison for this squash-input evaluation.
    saved = checkpoint["thresholds"].float().cpu()
    lb_saved = leaderboard_f1(probs, targets, saved)
    tuned, lb_full, lb_half = score_probs(probs, targets)
    print(f"checkpoint: {Path(path).name}  (saved epoch {checkpoint.get('epoch', '?')})")
    print(f"  macro-F1 with the saved thresholds           : {macro_f1(probs, targets, saved):.4f}")
    print(f"  leaderboard-F1 with the saved thresholds     : {lb_saved[0]:.4f} (P={lb_saved[1]:.4f}, R={lb_saved[2]:.4f})")
    print(f"  leaderboard-F1, thresholds re-tuned (optimistic): {lb_full:.4f}")
    print(f"  leaderboard-F1, half-split estimate          : {lb_half:.4f}")
    print("\nClasses that cost the most leaderboard points (saved thresholds):")
    per_class_report(probs, targets, saved)
    return probs, targets


if OLD_CHECKPOINT.exists():
    _ = evaluate_checkpoint(OLD_CHECKPOINT)
else:
    print(f"Previous checkpoint not found ({OLD_CHECKPOINT}); skipping the diagnosis.")


### 5.3 Interpreting the validation–test gap

The difference between validation performance and hidden-test performance was an important observation in both experimental branches.

In the ViT-Large experiments, an earlier checkpoint achieved approximately 0.80 standard macro-F1 on validation but around 0.72 on the hidden-test leaderboard.

These two scores are not directly comparable because standard macro-F1 and the official inverse-frequency-weighted F1 are different metrics.

The checkpoint diagnosis above therefore evaluates both metrics on the same validation predictions. It also distinguishes between scores obtained with saved thresholds, thresholds recalibrated on the full validation set, and half-split estimates.

We observed a related issue during the ensemble experiments:

| Approach | OOF leaderboard-style F1 | Hidden-test F1 |
|---|---:|---:|
| ConvNeXt + ViT-Base + ResNet50 ensemble | 0.7297 | 0.7018 |
| Ensemble + ML-Decoder | 0.7500 | 0.6863 |

Although ML-Decoder improved the out-of-fold validation estimate by approximately 0.0202, its hidden-test F1 decreased by 0.0155.

The five-fold results also showed greater variability for the ML-Decoder ensemble, suggesting that the apparent validation improvement was less stable.

These observations motivated a more cautious interpretation of threshold calibration and model selection.

In particular, we distinguish between:
- Standard macro-F1 and the official leaderboard-style metric.
- Scores evaluated on the same data used for calibration and held-out calibration estimates.
- Validation improvements and improvements confirmed on the hidden test set.

The final ViT-Large model was retained because it achieved the strongest reported hidden-test result, rather than relying solely on its validation score.


## 6. Training

### 6.1 Model, EMA, loss, optimiser and schedule

* **Loss:** `BCEWithLogitsLoss` with per-class positive weights computed from the training part only.
* **Optimiser:** AdamW with layer-wise learning-rate decay (timm builds one parameter group per layer depth with an `lr_scale`; bias / norm / position-embedding parameters get no weight decay).
* **Schedule:** one-cycle; warm-up over the first 15 % of the steps to the peak (`lr_scale` times `LR` per group), then smooth decay. Stepped after **every batch**.
* **Precision:** bfloat16 autocast. Unlike float16 it needs no loss scaling.

In [ ]:
# =============================================================================
# 6.1 TRIAL TRAINING FUNCTION
# =============================================================================
def run_one_trial(trial_lr: float, trial_layer_decay: float, trial_id: int):
    """Train one fixed 384+squash configuration and return its best result."""
    global BEST_CHECKPOINT

    seed_everything(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Re-create the model/EMA for every trial.
    print("\n" + "=" * 80)
    print(f"TRIAL {trial_id}: lr={trial_lr:.2e}, layer_decay={trial_layer_decay:.2f}")
    print("=" * 80)

    model = make_model(
        MODEL_NAME, device,
        resolution=RESOLUTION,
        drop_rate=DROP_RATE,
        drop_path_rate=DROP_PATH_RATE,
        pretrained=True,
    )
    ema = EMA(model, decay=EMA_DECAY)

    pos_weight = make_pos_weight(
        base_dataset.targets[train_indices],
        POS_WEIGHT_STRENGTH,
        POS_WEIGHT_CAP,
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="mean")

    optimizer = create_optimizer_v2(
        model,
        opt="adamw",
        lr=trial_lr,
        weight_decay=WEIGHT_DECAY,
        layer_decay=trial_layer_decay,
        betas=(0.9, 0.999),
    )

    max_lrs = [trial_lr * group.get("lr_scale", 1.0) for group in optimizer.param_groups]
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lrs,
        epochs=EPOCHS,
        steps_per_epoch=len(train_loader),
        pct_start=0.15,
        div_factor=25,
        final_div_factor=1000,
    )

    use_amp = device.type == "cuda"
    history = []
    trial_best = -1.0
    trial_best_epoch = -1
    trial_best_thresholds = None
    trial_best_tag = None
    trial_best_state = None

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels_for_loss = labels.to(device, non_blocking=True)
            labels_for_loss = labels_for_loss * (1 - LABEL_SMOOTHING) + LABEL_SMOOTHING / NUM_CLASSES

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=use_amp,
            ):
                logits = model(images)
                loss = criterion(logits.float(), labels_for_loss)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            ema.update(model)

            running_loss += loss.detach() * images.size(0)

        train_loss = running_loss.item() / (len(train_loader) * BATCH_SIZE)
        head_lr = max(group["lr"] for group in optimizer.param_groups)

        # Evaluate EMA only by default, matching the successful 384+squash setup.
        do_eval = epoch >= MIN_EVAL_EPOCH or epoch == EPOCHS
        record = {
            "trial_id": trial_id,
            "lr": trial_lr,
            "layer_decay": trial_layer_decay,
            "epoch": epoch,
            "train_loss": train_loss,
            "head_lr": head_lr,
        }

        if do_eval:
            probs, targets = predict_probs(
                ema.module, val_loader, device,
                amp=use_amp, tta=USE_TTA
            )
            thresholds, f1_full, f1_half = score_probs(probs, targets)
            macro = macro_f1(probs, targets, thresholds)

            record.update({
                "half_split_f1": f1_half,
                "full_f1": f1_full,
                "macro_f1": macro,
            })

            print(
                f"Trial {trial_id} | Epoch {epoch:02d}/{EPOCHS} | "
                f"loss={train_loss:.5f} | lr={head_lr:.2e} | "
                f"half-F1={f1_half:.4f} | full-F1={f1_full:.4f} | macro-F1={macro:.4f}"
            )

            if f1_half > trial_best:
                trial_best = f1_half
                trial_best_epoch = epoch
                trial_best_thresholds = thresholds.detach().cpu().clone()
                trial_best_tag = "EMA"
                # Snapshot EMA weights at this exact best epoch.
                trial_best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in ema.module.state_dict().items()
                }

        history.append(record)

    # Save the best checkpoint across ALL trials.
    if trial_best > run_one_trial.global_best:
        run_one_trial.global_best = trial_best
        BEST_CHECKPOINT = RUN_ROOT / "best_vit_large_384_lr_layer_decay_a100.pt"

        atomic_save({
            "state_dict": trial_best_state,
            "thresholds": trial_best_thresholds,
            "score": trial_best,
            "epoch": trial_best_epoch,
            "weights": trial_best_tag,
            "model_name": MODEL_NAME,
            "resolution": RESOLUTION,
            "data_config": data_config,
            "tta": USE_TTA,
            "eval_amp": use_amp,
            "validation_fraction": VALIDATION_FRACTION,
            "seed": SEED,
            "hyperparameters": {
                "lr": trial_lr,
                "weight_decay": WEIGHT_DECAY,
                "layer_decay": trial_layer_decay,
                "label_smoothing": LABEL_SMOOTHING,
                "pos_weight_strength": POS_WEIGHT_STRENGTH,
                "pos_weight_cap": POS_WEIGHT_CAP,
                "drop_rate": DROP_RATE,
                "drop_path_rate": DROP_PATH_RATE,
                "ema_decay": EMA_DECAY,
                "batch_size": BATCH_SIZE,
                "epochs": EPOCHS,
            },
        }, BEST_CHECKPOINT)

        print(f"  ★ GLOBAL BEST SAVED: {trial_best:.4f} -> {BEST_CHECKPOINT}")

    result = {
        "trial_id": trial_id,
        "lr": trial_lr,
        "layer_decay": trial_layer_decay,
        "best_epoch": trial_best_epoch,
        "best_half_split_f1": trial_best,
        "history": history,
    }
    del model, ema, optimizer, scheduler
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result

run_one_trial.global_best = -1.0


### 6.2 Training loop

Per epoch:

1. **Training pass** – binary label smoothing, bfloat16 forward pass, loss, backward pass, gradient clipping (norm 1.0), optimiser / scheduler / EMA updates.
2. **Validation** (from `MIN_EVAL_EPOCH` on, and always in the last epoch) – the EMA weights are evaluated (with flip averaging if `USE_TTA`); thresholds are tuned for the leaderboard-style metric and the **half-split score** is computed. The raw weights are evaluated only if `EVAL_RAW` is on or in the last epoch.
3. **Checkpointing** – if the half-split score is the best so far, the weights, the thresholds and the evaluation settings (TTA, precision) are saved to Drive. The evaluation settings are stored because the thresholds are only valid for predictions made the same way.
4. The epoch is appended to `training_history.json`.

In [ ]:
# =============================================================================
# 6.2 RUN LR × LAYER-DECAY SEARCH
# =============================================================================
trial_grid = [
    (lr, ld)
    for lr in LR_CANDIDATES
    for ld in LAYER_DECAY_CANDIDATES
]

all_results = []
all_histories = {}

print("Search grid:")
for i, (lr, ld) in enumerate(trial_grid, 1):
    print(f"  Trial {i}: lr={lr:.2e}, layer_decay={ld:.2f}")
print(f"Reference baseline: lr={BASELINE_LR:.2e}, layer_decay={BASELINE_LAYER_DECAY:.2f}, test F1={BASELINE_TEST_F1:.4f}")

for trial_id, (trial_lr, trial_layer_decay) in enumerate(trial_grid, 1):
    result = run_one_trial(trial_lr, trial_layer_decay, trial_id)
    all_results.append({
        "trial_id": result["trial_id"],
        "lr": result["lr"],
        "layer_decay": result["layer_decay"],
        "best_epoch": result["best_epoch"],
        "best_half_split_f1": result["best_half_split_f1"],
    })
    all_histories[str(trial_id)] = result["history"]

all_results = sorted(all_results, key=lambda x: x["best_half_split_f1"], reverse=True)
RESULTS_PATH.write_text(json.dumps(all_results, indent=2), encoding="utf-8")
HISTORY_PATH.write_text(json.dumps(all_histories, indent=2), encoding="utf-8")

print("\n" + "=" * 80)
print("LR × LAYER DECAY SEARCH FINISHED")
print("=" * 80)
for rank, r in enumerate(all_results, 1):
    print(
        f"{rank:>2}. lr={r['lr']:.2e} | layer_decay={r['layer_decay']:.2f} | "
        f"best epoch={r['best_epoch']} | val half-split F1={r['best_half_split_f1']:.5f}"
    )
print(f"\nBest checkpoint: {BEST_CHECKPOINT}")
print(f"Results JSON:    {RESULTS_PATH}")


### 6.3 Best-trial checkpoint

The selected configuration is the trial with the highest validation half-split F1 across the six LR × layer-decay combinations.

**Checkpointing detail:** when a new best epoch is observed, the notebook snapshots the EMA state and class thresholds at that point. The checkpoint therefore stores the weights and thresholds from the same selected epoch, rather than recording a best epoch while saving later weights.


In [ ]:
# =============================================================================
# 6.3 BEST TRIAL SUMMARY
# =============================================================================
best_trial = all_results[0]
print(json.dumps(best_trial, indent=2))
print(f"Reference 384+squash test F1: {BASELINE_TEST_F1:.4f}")
print(f"Best validation half-split F1: {best_trial['best_half_split_f1']:.5f}")


## 7. Test inference and submission JSON

`create_submission` rebuilds the network from the checkpoint (resolution, TTA and precision are read from it, so test predictions are made **exactly as the thresholds were tuned**), predicts the test images and writes a JSON file of the form

```json
{"000000000139": [56, 60, 62], "000000000285": [21], ...}
```

`check_submission` verifies the format before the file is uploaded.

> **Before uploading:** submitting again under the same group name replaces the previous leaderboard entry (see the course instructions).

In [ ]:
# =============================================================================
# 7. TEST INFERENCE AND SUBMISSION FILE
# =============================================================================
import torchvision.transforms as T
@torch.inference_mode()
def create_submission(data_root, checkpoint_path, json_path, *, batch_size=64, num_workers=8):
    """Predict the test images with a saved checkpoint and write the leaderboard JSON."""
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    use_tta = checkpoint.get("tta", False)             # older checkpoints: no TTA, float32
    amp = checkpoint.get("eval_amp", False)

    model = make_model(
        checkpoint["model_name"], dev,
        resolution=checkpoint["resolution"],
        drop_rate=checkpoint.get("hyperparameters", {}).get("drop_rate", 0.0),
        drop_path_rate=0.0,
        pretrained=False,                              # weights come from the checkpoint
    )
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    _, _, test_dir_ = dataset_paths(data_root)

    # IMPORTANT: test preprocessing must exactly match validation preprocessing.
    # Whole-image squash: resize the complete image directly to 384 x 384.
    test_size = checkpoint["data_config"]["input_size"][-1]
    test_mean = checkpoint["data_config"]["mean"]
    test_std = checkpoint["data_config"]["std"]
    test_transform = T.Compose([
        T.Resize(
            (test_size, test_size),
            interpolation=T.InterpolationMode.BICUBIC,
        ),
        T.ToTensor(),
        T.Normalize(mean=test_mean, std=test_std),
    ])

    loader = DataLoader(
        CocoTestDataset(test_dir_, test_transform), batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=dev.type == "cuda", persistent_workers=num_workers > 0,
    )

    thresholds = checkpoint["thresholds"].to(dev)
    predictions = {}
    for images, names in loader:
        images = images.to(dev, non_blocking=True)
        with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=amp and dev.type == "cuda"):
            probs = model(images).float().sigmoid()
            if use_tta:
                probs = 0.5 * (probs + model(images.flip(-1)).float().sigmoid())
        positive = probs >= thresholds                  # (batch, 80) booleans
        for name, row in zip(names, positive.cpu()):
            predictions[name] = torch.where(row)[0].tolist()

    json_path = Path(json_path)
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(predictions, indent=2))
    print(f"Wrote {len(predictions)} test predictions to {json_path} (TTA={use_tta}, bf16={amp})")
    return json_path


def check_submission(json_path, data_root):
    """Verify the format: one entry per test image, every value a list of ints in [0, 79]."""
    _, _, test_dir_ = dataset_paths(data_root)
    expected = {p.stem for p in Path(test_dir_).glob("*.jpg")}
    predictions = json.loads(Path(json_path).read_text())

    missing = expected - predictions.keys()
    extra = predictions.keys() - expected
    malformed = [
        k for k, v in predictions.items()
        if not (isinstance(v, list) and all(isinstance(i, int) and 0 <= i < NUM_CLASSES for i in v))
    ]
    empty = sum(1 for v in predictions.values() if len(v) == 0)

    print(f"entries: {len(predictions)} | test images: {len(expected)}")
    print(f"missing keys: {len(missing)} | unexpected keys: {len(extra)} | malformed values: {len(malformed)}")
    print(f"images with an empty prediction list: {empty}")
    assert not missing and not extra and not malformed, "Submission format check failed."
    print("Submission format OK.")

In [ ]:
# =============================================================================
# 7.1 CREATE AND CHECK THE SUBMISSION
# =============================================================================
assert BEST_CHECKPOINT.exists(), f"Checkpoint not found: {BEST_CHECKPOINT}"

SUBMISSION_JSON = create_submission(
    data_root=DATA_ROOT,
    checkpoint_path=BEST_CHECKPOINT,
    json_path=RUN_ROOT / "submission_vit_large_384_lr_layer_decay_a100.json",
    batch_size=64,
    num_workers=NUM_WORKERS,
)
check_submission(SUBMISSION_JSON, DATA_ROOT)


## 8. Final experiment record

### Selected checkpoint

| Field | Recorded value |
|---|---|
| Trial | **5** |
| Checkpoint file | `best_vit_large_384_lr_layer_decay_a100.pt` |
| Model | `vit_large_patch16_224.augreg_in21k_ft_in1k` |
| Resolution / preprocessing | 384 × 384 / whole-image squash |
| Learning rate | `1.5e-4` |
| Layer decay | `0.75` |
| Best epoch recorded in checkpoint | 3 |
| Validation half-split F1 recorded in checkpoint | 0.7423767447 |
| Weight type | EMA |
| EMA decay | 0.9996 |
| Weight decay | 0.05 |
| Batch size / configured epochs | 16 / 6 |
| Label smoothing | 0.10 |
| Positive-weight strength / cap | 0.5 / 20.0 |
| Drop rate / drop-path rate | 0.10 / 0.20 |
| Number of class-specific thresholds | 80 |
| Threshold range / mean | 0.13–0.85 / 0.7904 |

The metadata above was read from the saved checkpoint. It stores one threshold per class; the same thresholds are used by the inference function.


### 8.1 Consolidated hidden-test results

| Approach | Hidden-test F1 | Observation |
|---|---:|---|
| ConvNeXt-Small + ViT-Base ensemble | 0.6743 | Initial ensemble with test-time augmentation |
| Multi-view ConvNeXt + ViT ensemble | 0.6746 | Combined crop and letterbox predictions |
| ConvNeXt + ViT-Base + ResNet50 Focal | 0.7018 | Strong result in the early architecture/ensemble branch |
| ConvNeXt + ViT-Base + ResNet50 + ML-Decoder | 0.6863 | Better out-of-fold validation estimate but weaker hidden-test result |
| ViT-Large 384 × 384 — earlier version | 0.7255 | Center-crop test preprocessing |
| ViT-Large 384 × 384 — squash baseline | 0.7693 | Whole-image squash preprocessing |
| **ViT-Large 384 × 384 — reported final submission** | **0.7723** | **Best reported hidden-test result** |

### 8.2 Final leaderboard metrics

| Metric | Final result |
|---|---:|
| Accuracy | 0.8224 |
| Precision | 0.7483 |
| Recall | 0.7979 |
| **F1-score** | **0.7723** |

The reported final submission used the ViT-Large 384 × 384 squash approach. The selected LR × layer-decay search checkpoint metadata identifies Trial 5 (`lr=1.5e-4`, `layer_decay=0.75`, best epoch 3, EMA). The leaderboard score is an external hidden-test metric; it is recorded separately from validation scores.

### 8.3 Interpretation

- Whole-image squash preprocessing performed better than center cropping in the reported 384 × 384 comparison, consistent with retaining full-scene context and small objects.
- Within the six-trial search, LR `1.5e-4` and layer decay `0.75` achieved the highest validation half-split F1.
- Class-specific thresholds and TTA affect multi-label predictions, but validation improvements do not always transfer proportionally to the hidden test set.
- Validation half-split F1, full-validation F1, macro-F1 and leaderboard F1 are distinct metrics and should be labelled separately.



## 9. Comparative experiments

Before selecting the final ViT-Large model, we explored several pretrained architectures and training strategies on a local RTX 4060 GPU (8 GB VRAM).

All experiments in this branch used the same iterative multi-label stratified split (52,023 training images and 12,977 validation images, seed 42).

| Model | Training strategy | Validation macro-F1 |
|---|---|---:|
| ResNet50 | Frozen backbone, BCE | 0.6144 |
| ResNet50 | Fine-tuned layer4, BCE | 0.6771 |
| ResNet50 | Fine-tuned + threshold calibration | 0.6969 |
| ConvNeXt-Small | Full fine-tuning | 0.7350 |
| ConvNeXt-Small | Calibration + TTA | 0.7540 |
| ViT-Base/16 | Full fine-tuning with LLRD and EMA | 0.7543 |
| ViT-Base/16 | Calibration + TTA | 0.7718 |
| ConvNeXt + ViT-Base | Weighted ensemble + calibration | **0.7821** |

These results show three main trends:

- **Fine-tuning matters:** adapting pretrained features improved ResNet50 substantially compared with a frozen backbone.
- **Architecture matters:** ConvNeXt-Small and ViT-Base outperformed the initial ResNet50 baseline.
- **Calibration and ensembling matter:** class-specific thresholds and combining complementary architectures improved validation performance.

To investigate class imbalance, we also compared different loss functions using ResNet50.

| Loss function | Macro-F1 at threshold 0.5 |
|---|---:|
| BCE | **0.6771** |
| Weighted BCE | 0.6634 |
| Focal Loss | 0.6699 |

Neither Weighted BCE nor Focal Loss improved the standalone model over standard BCE in this setting.

However, the Focal Loss model later contributed useful diversity to the ensemble, illustrating that a weaker individual model can still improve combined predictions.

The strongest ensemble from this branch combined ConvNeXt-Small, ViT-Base and ResNet50 Focal, achieving a **hidden-test F1 of 0.7018**.

Adding ML-Decoder improved the out-of-fold validation estimate but reduced hidden-test F1 to 0.6863.

These experiments highlighted the importance of generalization: more complex models and higher validation scores do not necessarily produce better hidden-test results.
